# RETIA-DSS: train và test trên Google Colab

Notebook này clone code từ GitHub vào ổ đĩa nhanh của Colab, nhưng lưu checkpoint bền vững trong Google Drive. Dữ liệu đã có sẵn trong repo nên không cần upload riêng.

Trước khi chạy: chọn **Runtime > Change runtime type > T4 GPU** (hoặc GPU mạnh hơn), sau đó chạy từng cell từ trên xuống.

In [ ]:
!nvidia-smi


## 1. Cài môi trường tương thích

Mã RETIA gốc dùng PyTorch/DGL rất cũ. Cell dưới dùng PyTorch 2.4.1 + CUDA 12.4 và DGL 2.4.0, là cặp có wheel cho Python 3.12 của Colab. Không import `torch` trước khi cell này chạy xong.

In [ ]:
%pip uninstall -y torch torchvision torchaudio torchdata dgl
%pip install --no-cache-dir torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124
%pip install --no-cache-dir "dgl==2.4.0+cu124" -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
%pip install --no-cache-dir tqdm scipy pandas rdflib


In [ ]:
import sys, torch, dgl
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('DGL:', dgl.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Mount Drive, clone repo và nối thư mục checkpoint

Code và data chạy tại `/content/RETIA-DSS`; checkpoint trong `MyDrive/RETIA-DSS/models`. Nếu runtime mất kết nối, checkpoint vẫn còn trên Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_URL = 'https://github.com/DangHung1110/RETIA-DSS.git'
WORKSPACE = Path('/content/RETIA-DSS')
DRIVE_ROOT = Path('/content/drive/MyDrive/RETIA-DSS')
CHECKPOINT_DIR = DRIVE_ROOT / 'models'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not (WORKSPACE / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull', '--ff-only'], check=True)

models_path = WORKSPACE / 'models'
if models_path.is_symlink():
    models_path.unlink()
elif models_path.exists():
    shutil.copytree(models_path, CHECKPOINT_DIR, dirs_exist_ok=True)
    shutil.rmtree(models_path)
models_path.symlink_to(CHECKPOINT_DIR, target_is_directory=True)

print('Repo:', WORKSPACE)
print('Checkpoint:', models_path.resolve())
print('Data folders:', sorted(p.name for p in (WORKSPACE / 'data').iterdir() if p.is_dir()))


## 3. Kiểm tra nhanh trước khi train thật

Cell này chỉ chạy 2 epoch với mô hình nhỏ (`n_layers=1`, `n_hidden=32`) để bắt lỗi môi trường. Checkpoint của nó có tên khác mô hình chính. Có thể mất vài phút.

In [ ]:
smoke_cmd = [
    sys.executable, 'main.py', '-d', 'YAGO',
    '--train-history-len', '3', '--test-history-len', '3',
    '--lr', '0.001', '--n-layers', '1', '--n-hidden', '32',
    '--n-epochs', '2', '--evaluate-every', '1',
    '--self-loop', '--decoder', 'convtranse', '--encoder', 'uvrgcn',
    '--layer-norm', '--gpu', '0', '--ft_lr=0.001',
    '--norm_weight', '1', '--task-weight', '0.7'
]
subprocess.run(smoke_cmd, cwd=WORKSPACE / 'src', check=True)


## 4. Train mô hình chính

Bắt đầu với `YAGO` vì lệnh đơn giản, không cần static graph. Sau đó có thể đổi `DATASET` thành `WIKI`, `ICEWS14`, `ICEWS18` hoặc `ICEWS05-15`. Giữ nguyên toàn bộ tham số khi test để tên checkpoint khớp.

In [ ]:
DATASET = 'YAGO'  # @param ['YAGO', 'WIKI', 'ICEWS14', 'ICEWS18', 'ICEWS05-15']
N_EPOCHS = 250

DATASET_CONFIG = {
    'YAGO':       {'history': 3, 'extra': []},
    'WIKI':       {'history': 3, 'extra': []},
    'ICEWS14':    {'history': 9, 'extra': ['--weight', '0.5', '--angle', '14', '--discount', '1', '--add-static-graph']},
    'ICEWS18':    {'history': 4, 'extra': ['--weight', '0.5', '--angle', '10', '--discount', '1', '--add-static-graph']},
    'ICEWS05-15': {'history': 9, 'extra': ['--weight', '0.5', '--angle', '10', '--discount', '1', '--add-static-graph']},
}
cfg = DATASET_CONFIG[DATASET]

base_cmd = [
    sys.executable, 'main.py', '-d', DATASET,
    '--train-history-len', str(cfg['history']),
    '--test-history-len', str(cfg['history']),
    '--lr', '0.001', '--n-layers', '2', '--evaluate-every', '1',
    '--n-hidden', '200', '--self-loop', '--decoder', 'convtranse',
    '--encoder', 'uvrgcn', '--layer-norm', '--gpu', '0',
    '--ft_lr=0.001', '--norm_weight', '1', '--task-weight', '0.7',
] + cfg['extra']

train_cmd = base_cmd + ['--n-epochs', str(N_EPOCHS)]
print(' '.join(train_cmd))
subprocess.run(train_cmd, cwd=WORKSPACE / 'src', check=True)


## 5. Offline test

Chạy sau khi train xong. Kết quả cần ghi lại là `MRR` và `Hits@1/3/10`; ưu tiên dòng `filter_ent` khi báo cáo link prediction thực thể.

In [ ]:
test_cmd = base_cmd + ['--test']
print(' '.join(test_cmd))
subprocess.run(test_cmd, cwd=WORKSPACE / 'src', check=True)


## 6. Online continual training/test (tùy chọn)

Thứ tự đúng là: train tổng quát → `--test-valid` → `--test-test`. Bước online tạo nhiều checkpoint theo từng timestamp và tốn thêm dung lượng Drive.

In [ ]:
valid_cmd = base_cmd + ['--test-valid']
subprocess.run(valid_cmd, cwd=WORKSPACE / 'src', check=True)

online_test_cmd = base_cmd + ['--test-test']
subprocess.run(online_test_cmd, cwd=WORKSPACE / 'src', check=True)


In [ ]:
print('Các checkpoint đã lưu:')
for path in sorted(CHECKPOINT_DIR.rglob('*')):
    if path.is_file() and path.suffix != '.txt':
        print(path.relative_to(CHECKPOINT_DIR), f'{path.stat().st_size / 1024**2:.1f} MB')


## Lỗi thường gặp

- `CUDA available: False`: chưa chọn GPU runtime.
- `No matching distribution for dgl`: kiểm tra Python là 3.12 hoặc 3.11 và chạy lại cell cài đặt.
- Test lại chạy train: checkpoint không tồn tại hoặc tham số train/test không khớp nên tên file khác.
- Colab ngắt giữa chừng: chạy lại từ đầu; repo được clone lại nhưng checkpoint vẫn ở Drive. Code hiện tại chưa hỗ trợ resume giữa epoch, nên sẽ train lại general training từ epoch 0.
- Hết dung lượng Drive ở bước online: xóa các checkpoint theo timestamp không còn cần thiết, giữ lại checkpoint general tốt nhất.